In [ ]:
import pandas as pd
import duckdb

In [ ]:


MAX_LINE_SIZE = 20_000_000
MAX_REVIEW_LENGTH = 100_000

duckdb.sql(f"""
    COPY (
        SELECT *
        FROM read_csv(
            'all_reviews.csv',
            header = true,
            auto_detect = true,
            max_line_size = {MAX_LINE_SIZE}
        )
        WHERE length(review) <= {MAX_REVIEW_LENGTH}
    )
    TO 'reviews.parquet'
    (
        FORMAT PARQUET,
        COMPRESSION ZSTD
    )
""")

In [ ]:
df = pd.read_parquet("reviews.parquet")

In [ ]:


# ============================================================
# MAJOR PARAMETERS
# ============================================================

MIN_GAME_REVIEWS = 5   # Games must have at least this many reviews
MIN_USER_REVIEWS = 3   # Users must have at least this many remaining reviews

GAME_COLUMN = "appid"
USER_COLUMN = "author_steamid"

VERBOSE = True


# ============================================================
# FILTERING FUNCTION
# ============================================================

def iteratively_filter_reviews(
    dataframe: pd.DataFrame,
    min_game_reviews: int,
    min_user_reviews: int,
    game_column: str,
    user_column: str,
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Repeatedly removes:

    1. Games with fewer than min_game_reviews remaining reviews.
    2. Users with fewer than min_user_reviews remaining reviews.

    Continues until no additional rows are removed.
    """

    required_columns = {game_column, user_column}
    missing_columns = required_columns.difference(dataframe.columns)

    if missing_columns:
        raise ValueError(
            f"Missing required columns: {sorted(missing_columns)}"
        )

    if min_game_reviews < 1:
        raise ValueError("MIN_GAME_REVIEWS must be at least 1.")

    if min_user_reviews < 1:
        raise ValueError("MIN_USER_REVIEWS must be at least 1.")

    filtered_df = (
        dataframe
        .dropna(subset=[game_column, user_column])
        .copy()
    )

    iteration = 0

    while True:
        iteration += 1

        starting_rows = len(filtered_df)
        starting_games = filtered_df[game_column].nunique()
        starting_users = filtered_df[user_column].nunique()

        # Remove games below the threshold
        game_review_counts = filtered_df.groupby(
            game_column
        )[game_column].transform("size")

        filtered_df = filtered_df.loc[
            game_review_counts >= min_game_reviews
        ].copy()

        # Recalculate user counts after removing games
        user_review_counts = filtered_df.groupby(
            user_column
        )[user_column].transform("size")

        filtered_df = filtered_df.loc[
            user_review_counts >= min_user_reviews
        ].copy()

        ending_rows = len(filtered_df)
        ending_games = filtered_df[game_column].nunique()
        ending_users = filtered_df[user_column].nunique()

        if verbose:
            print(
                f"Iteration {iteration}: "
                f"rows {starting_rows:,} → {ending_rows:,} | "
                f"games {starting_games:,} → {ending_games:,} | "
                f"users {starting_users:,} → {ending_users:,}"
            )

        # Stop when an entire iteration removes nothing
        if ending_rows == starting_rows:
            break

    return filtered_df.reset_index(drop=True)


# ============================================================
# RUN
# ============================================================

processed_df = iteratively_filter_reviews(
    dataframe=df,
    min_game_reviews=MIN_GAME_REVIEWS,
    min_user_reviews=MIN_USER_REVIEWS,
    game_column=GAME_COLUMN,
    user_column=USER_COLUMN,
    verbose=VERBOSE,
)
# Faster write, somewhat larger file
processed_df.to_parquet(
    "processed_reviews.parquet",
    index=False,
    compression="snappy",
)

In [23]:
print(processed_df.shape)
print(df.shape)

(79844333, 24)
(113883557, 24)
